# AI Agents Workshop — Day 1 Labs (Colab)

S4DS KJSIT. Run the setup cell first, then work down.

**Before anything:** click the 🔑 key icon in the left sidebar → *Add new secret* →
name it exactly `HF_TOKEN` → paste your token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) →
toggle **Notebook access** on.

## Setup — run this once

In [ ]:
!pip install -q "huggingface_hub>=0.30.0" "smolagents>=1.14.0" "transformers>=4.45.0" duckduckgo-search

import os
from getpass import getpass

# Check environment variables first, then Google Colab secrets, and fallback to prompt
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not token:
    token = getpass("HF_TOKEN not found in env or Colab secrets. Enter token: ")

os.environ["HF_TOKEN"] = token
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"

from huggingface_hub import InferenceClient
client = InferenceClient(model=MODEL_ID, token=os.environ["HF_TOKEN"])
r = client.chat.completions.create(
    messages=[{"role": "user", "content": "Reply with exactly: pong"}],
    max_tokens=10,
)
print("model said:", r.choices[0].message.content)
print("SETUP OK" if "pong" in r.choices[0].message.content.lower() else "SETUP FAILED")

---
## Lab 1 — What the model actually sees

An LLM does not see a list of messages. It sees **one string**.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

messages = [
    {"role": "system", "content": "You are a terse assistant."},
    {"role": "user", "content": "What is the capital of Maharashtra?"},
    {"role": "assistant", "content": "Mumbai."},
    {"role": "user", "content": "And its population?"},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(repr(prompt))
print()
print(prompt)

**Try it:** swap in `meta-llama/Llama-3.2-1B-Instruct`. Completely different template.
This is why prompts don't transfer between models.

---
## Lab 2 — A tool is a function + a description

In [ ]:
import inspect

def get_weather(city: str) -> str:
    """Get the current weather for an Indian city.

    Args:
        city: Name of the city, e.g. "Pune".
        temp: Temperature in Celsius, e.g. "27C".
    """
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

def describe(func):
    sig = inspect.signature(func)
    doc = (func.__doc__ or "").strip().split("\n")[0]
    return f"- {func.__name__}{sig}: {doc}"

print(describe(get_weather))
print()
print("^ THIS is the only thing the model ever sees about your function.")

**Try it:** delete the docstring and re-run. Your docstring *is* your prompt.

---
## Lab 2.5 — Why bother with tools at all?

A tool is only worth its complexity if the model can't already answer.
So let's prove it: same question, asked twice.

1. **No tool** — the model answers from memory (frozen at training time).
2. **With a tool** — we search DuckDuckGo first and paste the results into the prompt.

Watch what changes.

In [ ]:
# Round 1 - no tool. The model answers from memory.

QUESTION = "What is the latest stable version of Python, and when was it released?"

def ask(prompt, max_tokens=250):
    r = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )
    return r.choices[0].message.content.strip()

no_tool = ask(QUESTION)

print("QUESTION:", QUESTION)
print("\n--- NO TOOL (from memory) ---")
print(no_tool)

In [ ]:
# Round 2 - search DuckDuckGo first. No API key is required.
from ddgs import DDGS

search_results = list(DDGS().text(QUESTION, max_results=5))

if not search_results:
    raise RuntimeError("DuckDuckGo returned no results. Try again or change the query.")

evidence = "\n\n".join(
    f"[{index}] {result.get('title', 'Untitled')}\n"
    f"{result.get('body', '')}\n"
    f"Source: {result.get('href', '')}"
    for index, result in enumerate(search_results, start=1)
 )

with_tool_prompt = f"""Answer the question using the fresh DuckDuckGo results below.
If the results do not contain enough evidence, say so. Include the source URLs.

Question: {QUESTION}

DuckDuckGo results:
{evidence}"""

with_tool = ask(with_tool_prompt)

print("\n--- WITH DUCKDUCKGO (live search, no API key) ---")
print(with_tool)
print("\n--- SOURCES ---")
for index, result in enumerate(search_results, start=1):
    print(f"[{index}] {result.get('href', '')}")

### Google's AI Overview as a tool

---
## Lab 3 — Write the agent loop yourself

The most important cell in this notebook. Read it line by line before running.

In [ ]:
import re

MAX_STEPS = 6

def get_weather(city: str) -> str:
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

def calculate(expression: str) -> str:
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: only numbers and + - * / ( ) allowed."
    try:
        return str(eval(expression))
    except Exception as exc:
        return f"Error: {exc}"

TOOLS = {"get_weather": get_weather, "calculate": calculate}

SYSTEM_PROMPT = """You solve tasks by reasoning step by step and using tools.

Available tools:
- get_weather(city): current weather for an Indian city.
- calculate(expression): evaluate arithmetic.

Reply in exactly this format, one step at a time:

Thought: <your reasoning>
Action: <tool_name>(<single argument>)

After each Action you will be shown an Observation.
When done, reply with:

Thought: <why you can answer now>
Final Answer: <your answer>

Never write an Observation yourself."""

ACTION_RE = re.compile(r"Action:\s*(\w+)\((.*?)\)", re.DOTALL)

def run(task):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, MAX_STEPS + 1):
        print(f"\n{'-'*50}\nSTEP {step}\n{'-'*50}")
        resp = client.chat.completions.create(
            messages=messages, max_tokens=400, stop=["Observation:"]
        )
        out = resp.choices[0].message.content.strip()
        print(out)
        messages.append({"role": "assistant", "content": out})

        if "Final Answer:" in out:
            return out.split("Final Answer:", 1)[1].strip()

        m = ACTION_RE.search(out)
        if not m:
            messages.append({"role": "user", "content":
                "Invalid format. Use 'Action: tool(arg)' or 'Final Answer: ...'."})
            continue

        name, arg = m.group(1), m.group(2).strip().strip("\"'")
        obs = TOOLS[name](arg) if name in TOOLS else f"Error: no tool '{name}'"
        print(f"\nObservation: {obs}")
        messages.append({"role": "user", "content": f"Observation: {obs}"})
    return "Gave up - hit MAX_STEPS."

print(run("What's the weather in Pune, and what is that temperature plus 5?"))

**Try it:** remove `stop=["Observation:"]` and re-run. The model invents its own
observations and confidently hallucinates. That one argument is the difference
between an agent and a liar.

---
## Lab 4 — The same thing with smolagents

In [ ]:
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for an Indian city.

    Use this whenever the user asks about temperature, rain, or humidity.

    Args:
        city: Name of the city, e.g. "Pune".
    """
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

@tool
def get_mess_menu(day: str) -> str:
    """Get the hostel mess menu for a day of the week.

    Args:
        day: Day name, e.g. "Tuesday".
    """
    menu = {"monday": "Rajma chawal", "tuesday": "Pav bhaji", "wednesday": "Veg biryani"}
    return menu.get(day.lower().strip(), f"No menu for {day}")

model = InferenceClientModel(model_id=MODEL_ID, token=os.environ["HF_TOKEN"])

agent = CodeAgent(tools=[get_weather, get_mess_menu], model=model, max_steps=5)
print(agent.run("Weather in Pune and Tuesday's mess menu - good day to eat outside?"))

**Try it:** add `verbosity_level=2` and compare the system prompt smolagents
generated against the one you wrote by hand in Lab 3.

### Three ways to make the same call — what actually changes

**A plain call** (no agent at all) is one round trip: you send messages, the
model replies, done. It cannot check the weather or read the mess menu —
it can only guess, because nothing exists to fetch real data mid-answer.

**smolagents (Lab 4)** turns that into a loop where the model's *action* is
literal Python. `agent.run()` sends the task, the model writes code like
`get_weather("Pune")`, smolagents executes it in a sandboxed interpreter,
and the result is fed back in. One call can chain several tool uses because
it's all one code block — that's why CodeAgent handles multi-step tasks well.

**LangChain / LangGraph (Lab 4.5)** runs the same Thought → Action →
Observation loop, but the action is a **JSON tool call**, not code — closer
to `ToolCallingAgent` from Lab 4 than to `CodeAgent`. `create_react_agent`
is LangGraph's prebuilt version of that loop: one tool call per step, each
one going back through the model before the next decision. It's the
framework most production agent stacks are actually built on, so the
concepts here — state graphs, checkpointing, tool binding — carry directly
into real jobs and larger systems in a way a hand-rolled loop doesn't scale to.

**Why this comparison matters:** all three are the *same idea* underneath —
model reasons, something acts, result comes back. The framework only changes
*how the action is encoded* and *how much infrastructure you get for free*.
Once you can see that, picking a framework becomes an engineering decision
(sandboxed code execution vs. structured tool calls vs. graph-based control
flow) instead of a taste preference.

| | Action format | Best for | You'll meet it |
|---|---|---|---|
| Plain call | none | no external data needed | everywhere, as a baseline |
| smolagents CodeAgent | Python | multi-step, chained tool use | fast prototyping, HF ecosystem |
| LangChain / LangGraph | JSON tool calls | production pipelines, complex control flow | most industry agent stacks |

In [ ]:
from langchain_core.tools import tool
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langgraph.prebuilt import create_react_agent

@tool
def get_weather(city: str) -> str:
    """Get the current weather for an Indian city.
    Use this whenever the user asks about temperature, rain, or humidity.
    Args:
        city: Name of the city, e.g. "Pune".
    """
    fake = {"mumbai": "32C, humid", "pune": "27C, clear", "delhi": "38C, hazy"}
    return fake.get(city.lower().strip(), f"No weather data for {city}")

@tool
def get_mess_menu(day: str) -> str:
    """Get the hostel mess menu for a day of the week.
    Args:
        day: Day name, e.g. "Tuesday".
    """
    menu = {"monday": "Rajma chawal", "tuesday": "Pav bhaji", "wednesday": "Veg biryani"}
    return menu.get(day.lower().strip(), f"No menu for {day}")

llm = HuggingFaceEndpoint(repo_id=MODEL_ID, huggingfacehub_api_token=os.environ["HF_TOKEN"])
chat_model = ChatHuggingFace(llm=llm)

langchain_agent = create_react_agent(chat_model, tools=[get_weather, get_mess_menu])

result = langchain_agent.invoke(
    {"messages": [("user", "Weather in Pune and Tuesday's mess menu - good day to eat outside?")]}
)
print(result["messages"][-1].content)